# 2. Agent, guard and approvals

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedelsayed741996-blip/berlin-data-portfolio/blob/main/b2b-saas-data-platforms/agentic-lakehouse-reference-architecture/notebooks/02_agent_guard_and_approvals.ipynb)

The agent loop asks a planner for the next step, passes every tool call through a guard and records spans. Offline the planner is a deterministic rule planner; in production it is Claude on Amazon Bedrock behind the same interface. Nothing here calls a model.

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

REPO = "https://github.com/mohammedelsayed741996-blip/berlin-data-portfolio.git"
PROJECT = "b2b-saas-data-platforms/agentic-lakehouse-reference-architecture"

if "google.colab" in sys.modules:
    if not Path("/content/berlin-data-portfolio").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/berlin-data-portfolio"], check=True)
    os.chdir(f"/content/berlin-data-portfolio/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jsonschema"], check=True)
else:
    here = Path.cwd()
    while not (here / "src" / "agentlake").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)
from IPython.display import Image, display
print("Project folder:", Path.cwd().name)

Project folder: agentic-lakehouse-reference-architecture


In [2]:
from agentlake import lakehouse
from agentlake.agent import Agent
from agentlake.config import Settings
from agentlake.tools import User
if not Path("lake/gold/account_ar.csv").exists():
    lakehouse.build_all(Path("lake"))
agent = Agent(Settings(trace_dir=Path("outputs/traces")))
am, fin, hof = User("AM02", "account_manager"), User("FIN01", "finance"), User("HOF01", "head_of_finance")
res = agent.run("What does Elbtal Maschinenbau owe us and how much is overdue?", am)
print(res.answer); print("tools:", [c["tool"] for c in res.calls])

Elbtal Maschinenbau GmbH (A002, owner AM02) has EUR 74,284.12 open, of which EUR 23,958.72 is overdue; the oldest open invoice is 19 days past due. Credit limit EUR 81,000.00, 92% used. Payment terms: 45 days.
tools: ['get_account']


The same question from another account manager gets the answer the agent would give for an account that does not exist. The guard never reveals that the account is someone else's.

In [3]:
other = User("AM05", "account_manager")
print(agent.run("What does Elbtal Maschinenbau owe us and how much is overdue?", other).answer)

No account found that you have access to.


## Actions wait for a person

Asking for a payment reminder creates an approval request. A credit hold needs the Head of Finance, and the person who proposed it cannot approve it.

In [4]:
r1 = agent.run("Draft a payment reminder for Elbtal Maschinenbau", am)
print(r1.answer)
from agentlake.approvals import ApprovalError
hold = agent.approvals.request("credit_hold", "A002", "AM02", "FIN01", {"reason": "over limit"})
for user, role in [("FIN01", "finance"), ("HOF01", "head_of_finance")]:
    try:
        a = agent.approvals.decide(hold.id, user, role, approve=True)
        print(f"{user}: {a.status}")
    except ApprovalError as e:
        print(f"{user}: refused ({e})")

I drafted a payment reminder (approval request #1). Nothing has been sent; you or Finance must approve it first.
FIN01: refused (role finance cannot decide on credit_hold)
HOF01: approved


## What a trace looks like

Each run is a tree of spans: the run, each planner step and each tool call with the guard's verdict and row count. Free text in arguments is truncated so traces do not collect personal data.

In [5]:
import json
res = agent.run("Which of my accounts are over their credit limit?", am, export=True)
spans = [json.loads(l) for l in open(f"outputs/traces/{res.trace_id}.jsonl")]
pd.DataFrame([{"span": s["name"], "duration_ms": s["duration_ms"], "status": s["status"],
               **{k: v for k, v in s["attributes"].items() if k in ("tool", "allowed", "rows", "step")}} for s in spans])

,span,duration_ms,status,step,tool,allowed,rows
0,agent.run,2.533,OK,NaN,NaN,NaN,NaN
1,planner.next,0.030,OK,0.0,NaN,NaN,NaN
2,tool.call,2.437,OK,0.0,list_accounts,True,1.0
3,planner.next,0.035,OK,1.0,NaN,NaN,NaN
